In [ ]:
import pandas as pd
import numpy as np
import cvxpy as cp
import math
import pickle
from itertools import chain, combinations


In [ ]:

with open('../data/processed/optimization_inputs_15_farmers.pkl', 'rb') as f:
    data = pickle.load(f)

# 2. Load the Standalone Values (Individual Rationality bounds)
with open('../data/processed/standalone_values_15_farmers.pkl', 'rb') as f:
    standalone_values = pickle.load(f)

# 3. Load the Characteristic Function (The heavy file you just computed!)
with open('../data/processed/v_dict_15_farmers_CF1.pkl', 'rb') as f:
    v_dict = pickle.load(f)

farmers = data['farmer_ids']
grand_coalition = tuple(farmers)
v_grand = v_dict[frozenset(grand_coalition)]

print(f"Loaded Characteristic Function with {len(v_dict)} sub-coalitions.")
print(f"Grand Coalition Value (v(S)): ₹ {v_grand:,.2f}")

In [ ]:
# Cell 2: Helper Functions
def powerset(iterable):
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s) + 1))

In [ ]:
# Cell 3: Shapley Value
def compute_shapley(v_dict, grand_coalition):
    n = len(grand_coalition)
    shapley_values = {player: 0.0 for player in grand_coalition}
    factorial_n = math.factorial(n)
    
    for i in grand_coalition:
        contribution = 0.0
        others = [p for p in grand_coalition if p != i]
        
        for subset in powerset(others):
            S = frozenset(subset)
            S_with_i = frozenset(list(S) + [i])
            
            weight = (math.factorial(len(S)) * math.factorial(n - len(S) - 1)) / factorial_n
            
            val_S = v_dict.get(S, 0.0)
            val_S_with_i = v_dict.get(S_with_i, 0.0)
            
            contribution += weight * (val_S_with_i - val_S)
            
        shapley_values[i] = contribution
    return shapley_values

print("Computing Shapley Value...")
shapley_alloc = compute_shapley(v_dict, grand_coalition)

In [ ]:
#  Least Core (CVXPY)
def compute_least_core(v_dict, grand_coalition, standalone_values):
    n = len(grand_coalition)
    players = list(grand_coalition)
    player_to_idx = {p: i for i, p in enumerate(players)}
    
    x = cp.Variable(n)
    epsilon = cp.Variable()
    
    constraints = []
    
    # Efficiency
    v_grand = v_dict[frozenset(players)]
    constraints.append(cp.sum(x) == v_grand)
    
    # Individual Rationality
    for p in players:
        idx = player_to_idx[p]
        constraints.append(x[idx] >= standalone_values[p])
        
    # Sub-coalition Rationality (Core Constraints)
    for subset in powerset(players):
        if 0 < len(subset) < n:
            S = frozenset(subset)
            indices = [player_to_idx[p] for p in S]
            val_S = v_dict.get(S, 0.0)
            constraints.append(cp.sum(x[indices]) >= val_S - epsilon)
            
    # Minimize maximum dissatisfaction
    objective = cp.Minimize(epsilon)
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.ECOS)
    
    if prob.status in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE]:
        allocations = {players[i]: x.value[i] for i in range(n)}
        return allocations, epsilon.value
    return None, None

print("Computing Least Core...")
least_core_alloc, epsilon_star = compute_least_core(v_dict, grand_coalition, standalone_values)
print(f"Least Core Epsilon (Instability): {epsilon_star:,.2f}")

In [ ]:
# Nucleolus
def compute_nucleolus(v_dict, grand_coalition, standalone_values):
    """
    Computes the nucleolus via sequential LP. Lexicographically minimizes
    the sorted vector of sub-coalition excesses.
    """
    n = len(grand_coalition)
    players = list(grand_coalition)
    player_to_idx = {p: i for i, p in enumerate(players)}
    
    # Exclude empty set and grand coalition from the inequality constraints
    all_subsets = [frozenset(s) for s in powerset(players) if 0 < len(s) < n]
    m = len(all_subsets)
    
    # Setup indicator matrix (A) and values (v) for coalitions
    A = np.zeros((m, n))
    v = np.zeros(m)
    for i, S in enumerate(all_subsets):
        v[i] = v_dict.get(S, 0.0)
        for p in S:
            A[i, player_to_idx[p]] = 1.0

    x = cp.Variable(n)
    epsilon = cp.Variable()
    
    active_constraints = []
    fixed_excesses = {}
    
    # 1. Efficiency & Individual Rationality
    v_grand = v_dict[frozenset(players)]
    active_constraints.append(cp.sum(x) == v_grand)
    for p in players:
        active_constraints.append(x[player_to_idx[p]] >= standalone_values[p])

    # 2. Iterative minimization of maximum excess
    while len(fixed_excesses) < m:
        current_constraints = active_constraints.copy()
        
        # Free constraints bounded by epsilon
        for i in range(m):
            if i not in fixed_excesses:
                current_constraints.append(v[i] - cp.sum(cp.multiply(A[i], x)) <= epsilon)
                
        prob = cp.Problem(cp.Minimize(epsilon), current_constraints)
        try:
            prob.solve(solver=cp.ECOS, warm_start=True)
        except Exception:
            break # Solver failed
            
        if prob.status not in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE]:
            break
            
        eps_val = epsilon.value
        
        # Find binding constraints (those hitting epsilon) and fix them for the next iteration
        new_fixed = False
        excesses = v - A @ x.value
        for i in range(m):
            if i not in fixed_excesses and np.isclose(excesses[i], eps_val, atol=1e-4):
                fixed_excesses[i] = eps_val
                # Lock this sub-coalition's excess at eps_val
                active_constraints.append(v[i] - cp.sum(cp.multiply(A[i], x)) == eps_val)
                new_fixed = True
                
        if not new_fixed:
            break # Avoid infinite loops if precision limits are reached
            
    allocations = {players[i]: x.value[i] for i in range(n)}
    return allocations

print("Computing Nucleolus...")
nucleolus_alloc = compute_nucleolus(v_dict, grand_coalition, standalone_values)
print("Nucleolus successfully computed.")

In [ ]:
# Naive Baselines (Equal Split & Area-Proportional Split)
def compute_naive_splits(v_grand, grand_coalition, data):
    n = len(grand_coalition)
    
    # 1. Equal Split
    equal_alloc = {p: v_grand / n for p in grand_coalition}
    
    # 2. Area-Proportional Split
    total_area = sum(data['farm_sizes'][p] for p in grand_coalition)
    prop_alloc = {p: (data['farm_sizes'][p] / total_area) * v_grand for p in grand_coalition}
    
    return equal_alloc, prop_alloc

equal_alloc, prop_alloc = compute_naive_splits(v_grand, grand_coalition, data)

In [ ]:
# Cell 6: Compile Results into a Table
results = []

for p in grand_coalition:
    results.append({
        'Farmer': p,
        'Farm Size (ha)': data['farm_sizes'][p],
        'Standalone Value (₹)': standalone_values[p],
        'Shapley Value (₹)': shapley_alloc[p],
        'Least Core (₹)': least_core_alloc[p] if least_core_alloc else np.nan,
        'Nucleolus (₹)': nucleolus_alloc[p] if nucleolus_alloc else np.nan,
        'Equal Split (₹)': equal_alloc[p],
        'Proportional Split (₹)': prop_alloc[p]
    })

df_results = pd.DataFrame(results)

# Add a totals row at the bottom to verify efficiency (all columns should sum to v(S))
totals = df_results.sum(numeric_only=True)
totals['Farmer'] = 'TOTAL'
df_results = pd.concat([df_results, pd.DataFrame([totals])], ignore_index=True)

# Formatting for display
format_cols = [
    'Standalone Value (₹)', 'Shapley Value (₹)', 'Least Core (₹)', 
    'Nucleolus (₹)', 'Equal Split (₹)', 'Proportional Split (₹)'
]

for col in format_cols:
    df_results[col] = df_results[col].map(lambda x: f"₹ {x:,.2f}" if pd.notnull(x) else "N/A")

display(df_results)

# Export to CSV for LaTeX
df_results.to_csv('../data/processed/final_allocations_15_farmers.csv', index=False)
print("Results exported to final_allocations_15_farmers.csv")

In [1]:
import gurobipy as gp

try:
    # Attempt to create an empty model
    m = gp.Model("test_model")
    print("Gurobi is successfully configured and the license is active.")
except gp.GurobiError as e:
    print(f"Gurobi Error {e.errno}: {e}")

ModuleNotFoundError: No module named 'gurobipy'